In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import your custom package
from cooltrack.constants import INDEPENDENT_DIMS
from cooltrack.data_loader import load_and_clean_grid_pandas
from cooltrack.models import ThermalEvolutionModels
from cooltrack.integrator import CoolingIntegrator
from cooltrack.initial_conditions import InitialConditions

sns.set_theme(style="whitegrid", context="talk")

# Paths
GRID_FILE_PATH = "../../data/HADES_grid/hades_processed_grid.parquet" 
AGE_DATA_PATH = "../../data/age_data/"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import your custom package
from cooltrack.constants import INDEPENDENT_DIMS
from cooltrack.data_loader import load_and_clean_grid_pandas
from cooltrack.models import ThermalEvolutionModels
from cooltrack.integrator import CoolingIntegrator
from cooltrack.initial_conditions import InitialConditions

sns.set_theme(style="whitegrid", context="talk")

# Paths
GRID_FILE_PATH = "../../data/HADES_grid/hades_processed_grid.parquet" 
AGE_DATA_PATH = "../../data/age_data/"

In [1]:
print("Using cooltrack.data_loader to fetch data...")
df_raw = load_and_clean_grid_pandas(GRID_FILE_PATH)

# Let's use a smaller slice of the data so training is instant for our plot test
df_test_raw = df_raw[df_raw['mass_Mj'] <= 10].copy()
print(f"Loaded {len(df_test_raw):,} rows.")

Using cooltrack.data_loader to fetch data...


NameError: name 'load_and_clean_grid_pandas' is not defined

In [ ]:
print("Training models and removing numerical noise...")
ml_engine = ThermalEvolutionModels()

# Catch the cleaned dataframe!
df_clean = ml_engine.train_models(df_test_raw, tune_hyperparameters=False, clean_outliers=True)

print(f"Models trained! Retained {len(df_clean):,} clean physics points.")

In [ ]:
print("Integrating a track using exact initial conditions...")

integrator = CoolingIntegrator(ml_engine)
init_cond = InitialConditions(AGE_DATA_PATH)

# Pick a target planet to plot (e.g., 1 Jupiter Mass, 500K Irradiation)
test_mass = 1.0
test_tirr = 500.0

# Find a matching row in our CLEAN dataset to get realistic fixed parameters
planet_row = df_clean[(np.isclose(df_clean['mass_Mj'], test_mass, atol=0.1)) & 
                      (np.isclose(df_clean['T_irr'], test_tirr, atol=10))].iloc[0]

# Get the exact hot-start entropy for a 1 Jupiter mass planet
s_hot_start = init_cond.get_starting_physical_entropy(test_mass, bin_index=19, n_bins=20)

# Integrate down to the coldest point in the clean grid
s_cold_end = df_clean['S_physical'].min()   

print(f"Integrating from S0 = {s_hot_start:.1f} down to {s_cold_end:.1f} J/K/kg...")

# Calculate the full track!
ages, entropies = integrator.calculate_track(planet_row, s_hot_start, s_cold_end)

if ages is not None:
    # Predict T_int at each step to plot the physical temperature
    fixed_params = planet_row[INDEPENDENT_DIMS].values.astype(float)
    t_ints = []
    
    for s in entropies:
        inp = np.append(fixed_params, s).reshape(1, -1)
        t_ints.append(ml_engine.tint_model.predict(inp)[0])

    # --- Plotting ---
    fig, ax1 = plt.subplots(figsize=(10, 6))

    color = 'tab:red'
    ax1.set_xlabel('Age (Years)')
    ax1.set_ylabel('Internal Temperature (K)', color=color)
    ax1.plot(ages, t_ints, color=color, lw=3)
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.set_xscale('log') 
    
    ax2 = ax1.twinx()  
    color = 'tab:blue'
    ax2.set_ylabel('Physical Entropy (J/K/kg)', color=color)  
    ax2.plot(ages, entropies, color=color, lw=3, linestyle='--')
    ax2.tick_params(axis='y', labelcolor=color)

    plt.title(f"CoolTrack: {test_mass} $M_J$, $T_{{irr}}$={test_tirr} K (Cleaned Models)")
    fig.tight_layout()  
    plt.show()
else:
    print("Integration failed or returned None.")

2026-02-23 02:23:56,164 - INFO - Performing first-pass training to identify grid outliers...


Training models using src.models.ThermalEvolutionModels...


2026-02-23 02:23:56,879 - INFO - Dropping 1822 corrupted grid points (> 1.0 dex error).
2026-02-23 02:23:56,899 - INFO - Training final T_int state model...
2026-02-23 02:23:58,687 - INFO - Final T_int R^2: 0.9665
2026-02-23 02:23:58,700 - INFO - Training final baseline dS/dt model...
2026-02-23 02:24:01,192 - INFO - Final dS/dt test R^2: 0.9957


Models trained and ready!
